# 🏀 NBA API Exploration Notebook

This notebook systematically explores every major endpoint of the `nba_api` library to catalog available features for building ML models to predict NBA win percentages.

**Models planned:**
1. **Team vs Team Model** — Capture playstyle matchups (pace, shooting profiles, defense schemes)
2. **Player Model** — Individual player impact that feeds into the team model
3. **Additional models** suggested at the end

**Approach:** For each endpoint, we call it with sample parameters, display `.head()` and `.dtypes`, then summarize useful features.

> ⚠️ The NBA API rate-limits requests. We add `time.sleep()` between calls. Some endpoints may occasionally timeout.

In [1]:
import pandas as pd
import time
from nba_api.stats.static import teams, players

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)

# Helper to safely call an endpoint and display info
def explore(name, df):
    """Display head and dtypes for a DataFrame returned by an nba_api endpoint."""
    print(f"\n{'='*80}")
    print(f"📊 {name}")
    print(f"   Shape: {df.shape}  |  Columns: {df.shape[1]}")
    print(f"{'='*80}")
    display(df.head(3))
    print(f"\n📋 Data Types:")
    display(df.dtypes)
    return df

## 1. Static Data (No API Call Required)
These are locally cached lookups — team IDs, player IDs, etc.

In [2]:
# --- Static: All NBA Teams ---
nba_teams = teams.get_teams()
df_teams = pd.DataFrame(nba_teams)
explore("Static Teams", df_teams)

print("\n")

# --- Static: All NBA Players ---
nba_players = players.get_players()
df_players = pd.DataFrame(nba_players)
explore("Static Players", df_players)


📊 Static Teams
   Shape: (30, 7)  |  Columns: 7


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970



📋 Data Types:


id              int64
full_name         str
abbreviation      str
nickname          str
city              str
state             str
year_founded    int64
dtype: object




📊 Static Players
   Shape: (5103, 5)  |  Columns: 5


,id,full_name,first_name,last_name,is_active
0,76001,Alaa Abdelnaby,Alaa,Abdelnaby,False
1,76002,Zaid Abdul-Aziz,Zaid,Abdul-Aziz,False
2,76003,Kareem Abdul-Jabbar,Kareem,Abdul-Jabbar,False



📋 Data Types:


id            int64
full_name       str
first_name      str
last_name       str
is_active      bool
dtype: object

,id,full_name,first_name,last_name,is_active
0,76001,Alaa Abdelnaby,Alaa,Abdelnaby,False
1,76002,Zaid Abdul-Aziz,Zaid,Abdul-Aziz,False
2,76003,Kareem Abdul-Jabbar,Kareem,Abdul-Jabbar,False
3,51,Mahmoud Abdul-Rauf,Mahmoud,Abdul-Rauf,False
4,1505,Tariq Abdul-Wahad,Tariq,Abdul-Wahad,False
...,...,...,...,...,...
5098,78650,Matt Zunic,Matt,Zunic,False
5099,1641783,Tristan da Silva,Tristan,da Silva,True
5100,1628427,Vlatko Čančar,Vlatko,Čančar,False
5101,1642365,Nikola Đurišić,Nikola,Đurišić,True


## 2. Team-Level Endpoints
Endpoints that return team-level stats — the core of the **Team vs Team model**.

We'll use the **Los Angeles Lakers (1610612747)** and season **2024-25** as sample parameters.

In [3]:
from nba_api.stats.endpoints import (
    teamgamelog, teamgamelogs, teamestimatedmetrics,
    teaminfocommon, teamdashboardbygeneralsplits, teamdashboardbyshootingsplits,
    teamdashlineups, teamdashptpass, teamdashptreb, teamdashptshots,
    teamplayerdashboard, teamplayeronoffdetails, teamplayeronoffsummary,
    teamyearbyyearstats, teamdetails,
    leaguedashteamstats, leaguedashteamclutch, leaguedashteamptshot,
    leaguedashteamshotlocations, leaguedashptteamdefend,
    leaguehustlestatsteam, leaguestandingsv3
)

TEAM_ID = 1610612747  # Lakers
SEASON = '2024-25'

# Store all explored DataFrames for reference
explored = {}

In [4]:
# 2a. TeamGameLog — per-game results for a team
tgl = teamgamelog.TeamGameLog(team_id=TEAM_ID, season=SEASON)
df = tgl.get_data_frames()[0]
explored['TeamGameLog'] = explore("TeamGameLog — Per-game W/L, points, FG%, etc.", df)

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [5]:
# 2b. TeamEstimatedMetrics — RPM-style estimated metrics per team
time.sleep(1)
tem = teamestimatedmetrics.TeamEstimatedMetrics(season=SEASON)
df = tem.get_data_frames()[0]
explored['TeamEstimatedMetrics'] = explore("TeamEstimatedMetrics — Estimated OFF/DEF ratings, pace, W%", df)


📊 TeamEstimatedMetrics — Estimated OFF/DEF ratings, pace, W%
   Shape: (30, 30)  |  Columns: 30


,TEAM_NAME,TEAM_ID,GP,W,L,W_PCT,MIN,E_OFF_RATING,E_DEF_RATING,E_NET_RATING,E_PACE,E_AST_RATIO,E_OREB_PCT,E_DREB_PCT,E_REB_PCT,E_TM_TOV_PCT,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,E_OFF_RATING_RANK,E_DEF_RATING_RANK,E_NET_RATING_RANK,E_AST_RATIO_RANK,E_OREB_PCT_RANK,E_DREB_PCT_RANK,E_REB_PCT_RANK,E_TM_TOV_PCT_RANK,E_PACE_RANK
0,Memphis Grizzlies,1610612763,82,48,34,0.585,3941.0,114.4,109.9,4.6,106.2,19.2,0.287,0.749,0.520,0.147,1,11,11,11,29,6,9,6,12,3,14,2,23,1
1,Chicago Bulls,1610612741,82,39,43,0.476,3951.0,111.9,113.4,-1.5,104.9,20.1,0.223,0.766,0.499,0.139,1,18,18,18,18,17,18,18,6,28,4,18,15,2
2,Atlanta Hawks,1610612737,82,40,42,0.488,3956.0,111.9,112.9,-1.1,105.1,20.1,0.263,0.760,0.505,0.147,1,16,16,16,14,18,19,19,7,10,9,12,22,3



📋 Data Types:


TEAM_NAME                str
TEAM_ID                int64
GP                     int64
W                      int64
L                      int64
W_PCT                float64
MIN                  float64
E_OFF_RATING         float64
E_DEF_RATING         float64
E_NET_RATING         float64
E_PACE               float64
E_AST_RATIO          float64
E_OREB_PCT           float64
E_DREB_PCT           float64
E_REB_PCT            float64
E_TM_TOV_PCT         float64
GP_RANK                int64
W_RANK                 int64
L_RANK                 int64
W_PCT_RANK             int64
MIN_RANK               int64
E_OFF_RATING_RANK      int64
E_DEF_RATING_RANK      int64
E_NET_RATING_RANK      int64
E_AST_RATIO_RANK       int64
E_OREB_PCT_RANK        int64
E_DREB_PCT_RANK        int64
E_REB_PCT_RANK         int64
E_TM_TOV_PCT_RANK      int64
E_PACE_RANK            int64
dtype: object

In [6]:
# 2c. LeagueDashTeamStats — comprehensive team stats (Base, Advanced, Four Factors, etc.)
time.sleep(1)

# Base stats
ldts_base = leaguedashteamstats.LeagueDashTeamStats(season=SEASON, measure_type_detailed_defense='Base')
df = ldts_base.get_data_frames()[0]
explored['LeagueDashTeamStats_Base'] = explore("LeagueDashTeamStats (Base) — Traditional team stats", df)

time.sleep(1)

# Advanced stats (OFF_RATING, DEF_RATING, NET_RATING, PACE, PIE, etc.)
ldts_adv = leaguedashteamstats.LeagueDashTeamStats(season=SEASON, measure_type_detailed_defense='Advanced')
df = ldts_adv.get_data_frames()[0]
explored['LeagueDashTeamStats_Advanced'] = explore("LeagueDashTeamStats (Advanced) — OFF/DEF Rating, Pace, eFG%, TS%", df)

time.sleep(1)

# Four Factors 
ldts_ff = leaguedashteamstats.LeagueDashTeamStats(season=SEASON, measure_type_detailed_defense='Four Factors')
df = ldts_ff.get_data_frames()[0]
explored['LeagueDashTeamStats_FourFactors'] = explore("LeagueDashTeamStats (Four Factors) — eFG%, TOV%, OREB%, FTA Rate", df)

time.sleep(1)

# Opponent stats
ldts_opp = leaguedashteamstats.LeagueDashTeamStats(season=SEASON, measure_type_detailed_defense='Opponent')
df = ldts_opp.get_data_frames()[0]
explored['LeagueDashTeamStats_Opponent'] = explore("LeagueDashTeamStats (Opponent) — What opponents shoot against this team", df)


📊 LeagueDashTeamStats (Base) — Traditional team stats
   Shape: (30, 54)  |  Columns: 54


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,82,40,42,0.488,3956.0,3556,7528,0.472,1107,3089,0.358,1472,1900,0.775,974,2675,3649,2426,1273.0,798,419,403,1564,1563,9691,-93.0,1,16,16,16,14,6,5,14,14,15,18,3,4,18,9,23,14,2,26,2,12,15,22,8,5,18
1,1610612738,Boston Celtics,82,61,21,0.744,3966.0,3413,7382,0.462,1457,3955,0.368,1251,1565,0.799,933,2783,3716,2144,973.0,589,452,295,1303,1438,9534,747.0,1,3,3,3,6,16,11,19,1,1,10,30,30,6,10,7,8,16,2,29,5,1,2,27,8,3
2,1610612751,Brooklyn Nets,82,26,56,0.317,3956.0,3086,7060,0.437,1113,3232,0.344,1334,1696,0.787,894,2493,3387,2063,1244.0,639,352,457,1698,1550,8619,-583.0,1,25,25,25,14,30,28,29,13,8,25,23,23,11,18,29,29,24,23,20,26,27,28,14,29,26



📋 Data Types:


TEAM_ID              int64
TEAM_NAME              str
GP                   int64
W                    int64
L                    int64
W_PCT              float64
MIN                float64
FGM                  int64
FGA                  int64
FG_PCT             float64
FG3M                 int64
FG3A                 int64
FG3_PCT            float64
FTM                  int64
FTA                  int64
FT_PCT             float64
OREB                 int64
DREB                 int64
REB                  int64
AST                  int64
TOV                float64
STL                  int64
BLK                  int64
BLKA                 int64
PF                   int64
PFD                  int64
PTS                  int64
PLUS_MINUS         float64
GP_RANK              int64
W_RANK               int64
L_RANK               int64
W_PCT_RANK           int64
MIN_RANK             int64
FGM_RANK             int64
FGA_RANK             int64
FG_PCT_RANK          int64
FG3M_RANK            int64
F


📊 LeagueDashTeamStats (Advanced) — OFF/DEF Rating, Pace, eFG%, TS%
   Shape: (30, 46)  |  Columns: 46


...

TEAM_ID                int64
TEAM_NAME                str
GP                     int64
W                      int64
L                      int64
W_PCT                float64
MIN                  float64
EFG_PCT              float64
FTA_RATE             float64
TM_TOV_PCT           float64
OREB_PCT             float64
OPP_EFG_PCT          float64
OPP_FTA_RATE         float64
OPP_TOV_PCT          float64
OPP_OREB_PCT         float64
GP_RANK                int64
W_RANK                 int64
L_RANK                 int64
W_PCT_RANK             int64
MIN_RANK               int64
EFG_PCT_RANK           int64
FTA_RATE_RANK          int64
TM_TOV_PCT_RANK        int64
OREB_PCT_RANK          int64
OPP_EFG_PCT_RANK       int64
OPP_FTA_RATE_RANK      int64
OPP_TOV_PCT_RANK       int64
OPP_OREB_PCT_RANK      int64
dtype: object


📊 LeagueDashTeamStats (Opponent) — What opponents shoot against this team
   Shape: (30, 54)  |  Columns: 54


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,OPP_FGM_RANK,OPP_FGA_RANK,OPP_FG_PCT_RANK,OPP_FG3M_RANK,OPP_FG3A_RANK,OPP_FG3_PCT_RANK,OPP_FTM_RANK,OPP_FTA_RANK,OPP_FT_PCT_RANK,OPP_OREB_RANK,OPP_DREB_RANK,OPP_REB_RANK,OPP_AST_RANK,OPP_TOV_RANK,OPP_STL_RANK,OPP_BLK_RANK,OPP_BLKA_RANK,OPP_PF_RANK,OPP_PFD1,OPP_PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,82,40,42,0.488,3956.0,3562.0,7398.0,0.481,1169.0,3098.0,0.377,1491.0,1858.0,0.802,845.0,2735.0,3580.0,2315.0,1315.0,755.0,403.0,419.0,1563.0,1564.0,9784.0,93.0,1,16,16,16,14,25,20,28,25,16,28,24,23,30,5,18,12,24,3,28,15,12,23,9,27,18
1,1610612738,Boston Celtics,82,61,21,0.744,3966.0,3302.0,7331.0,0.450,1055.0,3041.0,0.347,1128.0,1451.0,0.777,878.0,2704.0,3582.0,1968.0,1048.0,567.0,295.0,452.0,1438.0,1303.0,8787.0,-747.0,1,3,3,3,6,4,17,2,6,11,3,1,1,14,9,15,13,3,27,2,1,5,4,29,2,3
2,1610612751,Brooklyn Nets,82,26,56,0.317,3956.0,3317.0,6909.0,0.480,1059.0,2928.0,0.362,1509.0,1936.0,0.779,829.0,2754.0,3583.0,2219.0,1228.0,660.0,457.0,352.0,1550.0,1698.0,9202.0,583.0,1,25,25,25,14,6,2,27,8,6,17,26,25,18,4,21,14,19,12,13,27,26,17,3,11,26



📋 Data Types:


TEAM_ID               int64
TEAM_NAME               str
GP                    int64
W                     int64
L                     int64
W_PCT               float64
MIN                 float64
OPP_FGM             float64
OPP_FGA             float64
OPP_FG_PCT          float64
OPP_FG3M            float64
OPP_FG3A            float64
OPP_FG3_PCT         float64
OPP_FTM             float64
OPP_FTA             float64
OPP_FT_PCT          float64
OPP_OREB            float64
OPP_DREB            float64
OPP_REB             float64
OPP_AST             float64
OPP_TOV             float64
OPP_STL             float64
OPP_BLK             float64
OPP_BLKA            float64
OPP_PF              float64
OPP_PFD             float64
OPP_PTS             float64
PLUS_MINUS          float64
GP_RANK               int64
W_RANK                int64
L_RANK                int64
W_PCT_RANK            int64
MIN_RANK              int64
OPP_FGM_RANK          int64
OPP_FGA_RANK          int64
OPP_FG_PCT_RANK     

In [7]:
# 2d. LeagueDashTeamClutch — team performance in clutch situations
time.sleep(1)
ldtc = leaguedashteamclutch.LeagueDashTeamClutch(season=SEASON)
df = ldtc.get_data_frames()[0]
explored['LeagueDashTeamClutch'] = explore("LeagueDashTeamClutch — Clutch stats (last 5 min, ±5 pts)", df)


📊 LeagueDashTeamClutch — Clutch stats (last 5 min, ±5 pts)
   Shape: (30, 54)  |  Columns: 54


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,41,21,20,0.512,160.035000,125,277,0.451,37,109,0.339,115,141,0.816,39,110,149,79,43.0,22,17,16,80,99,402,29.0,7,8,17,15,6,6,6,10,7,12,11,1,2,5,10,5,7,2,21,9,10,16,14,4,3,7
1,1610612738,Boston Celtics,35,24,11,0.686,138.858333,118,246,0.480,50,132,0.379,79,104,0.760,33,84,117,68,32.0,17,20,7,69,83,365,59.0,21,4,2,1,16,8,13,3,1,3,6,17,17,22,17,18,21,5,7,22,4,3,8,12,7,3
2,1610612751,Brooklyn Nets,38,17,21,0.447,145.363333,98,234,0.419,39,111,0.351,69,86,0.802,35,94,129,60,53.0,14,12,20,107,65,304,-49.0,12,19,22,23,13,19,17,21,6,9,9,21,24,11,14,11,14,12,28,26,22,24,30,23,19,26



📋 Data Types:


TEAM_ID              int64
TEAM_NAME              str
GP                   int64
W                    int64
L                    int64
W_PCT              float64
MIN                float64
FGM                  int64
FGA                  int64
FG_PCT             float64
FG3M                 int64
FG3A                 int64
FG3_PCT            float64
FTM                  int64
FTA                  int64
FT_PCT             float64
OREB                 int64
DREB                 int64
REB                  int64
AST                  int64
TOV                float64
STL                  int64
BLK                  int64
BLKA                 int64
PF                   int64
PFD                  int64
PTS                  int64
PLUS_MINUS         float64
GP_RANK              int64
W_RANK               int64
L_RANK               int64
W_PCT_RANK           int64
MIN_RANK             int64
FGM_RANK             int64
FGA_RANK             int64
FG_PCT_RANK          int64
FG3M_RANK            int64
F

In [8]:
# 2e. LeagueHustleStatsTeam — deflections, loose balls, contested shots, charges
time.sleep(1)
lhst = leaguehustlestatsteam.LeagueHustleStatsTeam(season=SEASON)
df = lhst.get_data_frames()[0]
explored['LeagueHustleStatsTeam'] = explore("LeagueHustleStatsTeam — Hustle stats (deflections, loose balls, etc.)", df)


📊 LeagueHustleStatsTeam — Hustle stats (deflections, loose balls, etc.)
   Shape: (30, 20)  |  Columns: 20


,TEAM_ID,TEAM_NAME,MIN,CONTESTED_SHOTS,CONTESTED_SHOTS_2PT,CONTESTED_SHOTS_3PT,DEFLECTIONS,CHARGES_DRAWN,SCREEN_ASSISTS,SCREEN_AST_PTS,OFF_LOOSE_BALLS_RECOVERED,DEF_LOOSE_BALLS_RECOVERED,LOOSE_BALLS_RECOVERED,PCT_LOOSE_BALLS_RECOVERED_OFF,PCT_LOOSE_BALLS_RECOVERED_DEF,OFF_BOXOUTS,DEF_BOXOUTS,BOX_OUTS,PCT_BOX_OUTS_OFF,PCT_BOX_OUTS_DEF
0,1610612737,Atlanta Hawks,3927.0,3386,2015,1371,1563,34,598,1414,193,200,393,0.491,0.509,96,358,454,0.211,0.789
1,1610612738,Boston Celtics,3930.0,3555,2161,1394,1155,19,685,1746,206,187,393,0.524,0.476,122,459,581,0.210,0.790
2,1610612751,Brooklyn Nets,3923.0,3057,1741,1316,1314,41,598,1392,183,170,353,0.518,0.482,87,388,475,0.183,0.817



📋 Data Types:


TEAM_ID                            int64
TEAM_NAME                            str
MIN                              float64
CONTESTED_SHOTS                    int64
CONTESTED_SHOTS_2PT                int64
CONTESTED_SHOTS_3PT                int64
DEFLECTIONS                        int64
CHARGES_DRAWN                      int64
SCREEN_ASSISTS                     int64
SCREEN_AST_PTS                     int64
OFF_LOOSE_BALLS_RECOVERED          int64
DEF_LOOSE_BALLS_RECOVERED          int64
LOOSE_BALLS_RECOVERED              int64
PCT_LOOSE_BALLS_RECOVERED_OFF    float64
PCT_LOOSE_BALLS_RECOVERED_DEF    float64
OFF_BOXOUTS                        int64
DEF_BOXOUTS                        int64
BOX_OUTS                           int64
PCT_BOX_OUTS_OFF                 float64
PCT_BOX_OUTS_DEF                 float64
dtype: object

In [9]:
# 2f. LeagueStandingsV3 — conference/division standings, streaks, last 10
time.sleep(1)
ls = leaguestandingsv3.LeagueStandingsV3(season=SEASON)
df = ls.get_data_frames()[0]
explored['LeagueStandingsV3'] = explore("LeagueStandingsV3 — Full standings (W/L, streak, L10, home/road)", df)

time.sleep(1)

# 2g. LeagueDashTeamPtShot — team shooting by shot type (catch & shoot, pull-up, etc.)
ldtps = leaguedashteamptshot.LeagueDashTeamPtShot(season=SEASON)
df = ldtps.get_data_frames()[0]
explored['LeagueDashTeamPtShot'] = explore("LeagueDashTeamPtShot — Catch & shoot, pull-up, paint touch shooting", df)

time.sleep(1)

# 2h. LeagueDashPtTeamDefend — opponent shot defense by zone/distance
ldptd = leaguedashptteamdefend.LeagueDashPtTeamDefend(season=SEASON, defense_category='Overall')
df = ldptd.get_data_frames()[0]
explored['LeagueDashPtTeamDefend'] = explore("LeagueDashPtTeamDefend — Shot defense (contested %, DFG%)", df)

time.sleep(1)

# 2i. TeamDashboardByGeneralSplits — home/away, pre/post all-star, by month, etc.
tdgs = teamdashboardbygeneralsplits.TeamDashboardByGeneralSplits(team_id=TEAM_ID, season=SEASON)
dfs = tdgs.get_data_frames()
explored['TeamDashboardByGeneralSplits_Overall'] = explore("TeamDashboardByGeneralSplits — Overall", dfs[0])
explored['TeamDashboardByGeneralSplits_ByLocation'] = explore("TeamDashboardByGeneralSplits — Home/Away Split", dfs[1])


📊 LeagueStandingsV3 — Full standings (W/L, streak, L10, home/road)
   Shape: (30, 92)  |  Columns: 92


,LeagueID,SeasonID,TeamID,TeamCity,TeamName,TeamSlug,Conference,ConferenceRecord,PlayoffRank,ClinchIndicator,Division,DivisionRecord,DivisionRank,WINS,LOSSES,WinPCT,LeagueRank,Record,HOME,ROAD,L10,Last10Home,Last10Road,OT,ThreePTSOrLess,TenPTSOrMore,LongHomeStreak,strLongHomeStreak,LongRoadStreak,strLongRoadStreak,LongWinStreak,LongLossStreak,CurrentHomeStreak,strCurrentHomeStreak,CurrentRoadStreak,strCurrentRoadStreak,CurrentStreak,strCurrentStreak,ConferenceGamesBack,DivisionGamesBack,ClinchedConferenceTitle,ClinchedDivisionTitle,ClinchedPlayoffBirth,ClinchedPlayIn,EliminatedConference,EliminatedDivision,AheadAtHalf,BehindAtHalf,TiedAtHalf,AheadAtThird,BehindAtThird,TiedAtThird,Score100PTS,OppScore100PTS,OppOver500,LeadInFGPCT,LeadInReb,FewerTurnovers,PointsPG,OppPointsPG,DiffPointsPG,vsEast,vsAtlantic,vsCentral,vsSoutheast,vsWest,vsNorthwest,vsPacific,vsSouthwest,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Score_80_Plus,Opp_Score_80_Plus,Score_Below_80,Opp_Score_Below_80,TotalPoints,OppTotalPoints,DiffTotalPoints,LeagueGamesBack,PlayoffSeeding,ClinchedPostSeason,NEUTRAL
0,00,22024,1610612739,Cleveland,Cavaliers,cavaliers,East,41-11,1,- e,Central,12-4,1,64,18,0.780,NaN,64-18,34-7,30-11,6-4,7-3,5-5,1-1,5-4,41-7,10,W 10,11,W 11,16,4,-1,L 1,1,W 1,-1,L 1,0.0,0.0,1,1,1,0,0,0,50-9,11-9,3-0,55-3,9-15,0-0,64-17,56-18,29-12,52-1,41-4,34-11,121.9,112.4,9.5,41-11,16-3,12-4,13-4,23-7,9-1,6-4,8-2,10-5,10-1,11-5,4-3,0-0,0-0,0-0,0-0,0-0,5-0,12-3,12-1,64-18,64-18,0-0,0-0,9999,9217,782,4.0,1.0,1,0-0
1,00,22024,1610612760,Oklahoma City,Thunder,thunder,West,39-13,1,- w,Northwest,12-4,1,68,14,0.829,1.0,68-14,35-6,32-8,8-2,8-2,9-1,0-1,1-4,54-5,12,W 12,11,W 11,15,2,1,W 1,3,W 3,4,W 4,0.0,0.0,1,1,1,0,0,0,53-5,14-9,1-0,61-2,6-10,1-2,67-12,49-14,36-12,58-3,35-3,55-12,120.5,107.6,12.9,29-1,10-0,9-1,10-0,39-13,12-4,13-3,14-6,10-4,11-2,15-1,5-2,0-0,0-0,0-0,0-0,0-0,4-0,11-4,12-1,68-14,68-14,0-0,0-0,9881,8826,1055,0.0,1.0,1,1-0
2,00,22024,1610612738,Boston,Celtics,celtics,East,39-13,2,- a,Atlantic,14-2,1,61,21,0.744,NaN,61-21,28-13,33-8,8-2,8-2,9-1,4-2,8-3,41-7,7,W 7,9,W 9,9,2,4,W 4,-1,L 1,2,W 2,3.0,0.0,0,1,1,0,0,0,50-6,8-14,3-1,56-7,4-10,1-4,60-15,42-20,25-12,42-3,40-6,39-8,116.3,107.2,9.1,39-13,14-2,12-6,13-5,22-8,8-2,7-3,7-3,10-6,8-3,14-1,5-2,0-0,0-0,0-0,0-0,0-0,4-1,12-2,8-6,61-20,60-21,0-1,1-0,9534,8787,747,7.0,2.0,1,0-0



📋 Data Types:


LeagueID                  str
SeasonID                  str
TeamID                  int64
TeamCity                  str
TeamName                  str
                       ...   
DiffTotalPoints         int64
LeagueGamesBack       float64
PlayoffSeeding        float64
ClinchedPostSeason      int64
NEUTRAL                   str
Length: 92, dtype: object


📊 LeagueDashTeamPtShot — Catch & shoot, pull-up, paint touch shooting
   Shape: (30, 18)  |  Columns: 18


...

GROUP_SET              str
GROUP_VALUE            str
SEASON_YEAR            str
GP                   int64
W                    int64
L                    int64
W_PCT              float64
MIN                float64
FGM                  int64
FGA                  int64
FG_PCT             float64
FG3M                 int64
FG3A                 int64
FG3_PCT            float64
FTM                  int64
FTA                  int64
FT_PCT             float64
OREB                 int64
DREB                 int64
REB                  int64
AST                  int64
TOV                float64
STL                  int64
BLK                  int64
BLKA                 int64
PF                   int64
PFD                  int64
PTS                  int64
PLUS_MINUS         float64
GP_RANK              int64
W_RANK               int64
L_RANK               int64
W_PCT_RANK           int64
MIN_RANK             int64
FGM_RANK             int64
FGA_RANK             int64
FG_PCT_RANK          int64
F


📊 TeamDashboardByGeneralSplits — Home/Away Split
   Shape: (2, 55)  |  Columns: 55


,GROUP_SET,GROUP_VALUE,TEAM_GAME_LOCATION,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,Location,Home,Home,41,31,10,0.756,1973.0,1691,3488,0.485,569,1533,0.371,772,995,0.776,396,1352,1748,1093,568.0,325,191,159,706,810,4723,198.0,1,1,1,1,1,1,2,1,1,1,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1
1,Location,Road,Road,41,19,22,0.463,1973.0,1665,3519,0.473,523,1448,0.361,722,907,0.796,396,1335,1731,1038,577.0,308,176,185,709,766,4575,-98.0,1,2,2,2,1,2,1,2,2,2,2,2,2,1,1,2,2,2,2,2,2,2,2,2,2,2



📋 Data Types:


GROUP_SET                 str
GROUP_VALUE               str
TEAM_GAME_LOCATION        str
GP                      int64
W                       int64
L                       int64
W_PCT                 float64
MIN                   float64
FGM                     int64
FGA                     int64
FG_PCT                float64
FG3M                    int64
FG3A                    int64
FG3_PCT               float64
FTM                     int64
FTA                     int64
FT_PCT                float64
OREB                    int64
DREB                    int64
REB                     int64
AST                     int64
TOV                   float64
STL                     int64
BLK                     int64
BLKA                    int64
PF                      int64
PFD                     int64
PTS                     int64
PLUS_MINUS            float64
GP_RANK                 int64
W_RANK                  int64
L_RANK                  int64
W_PCT_RANK              int64
MIN_RANK  

In [10]:
# 2j. SynergyPlayTypes — playstyle breakdown (Isolation, PnR, Transition, etc.) for TEAMS
from nba_api.stats.endpoints import synergyplaytypes, leaguedashptstats, leaguedashlineups, leagueseasonmatchups

time.sleep(1)
spt = synergyplaytypes.SynergyPlayTypes(season=SEASON, player_or_team_abbreviation='T', type_grouping_nullable='offensive')
df = spt.get_data_frames()[0]
explored['SynergyPlayTypes_Team_Off'] = explore("SynergyPlayTypes (Team, Offense) — Iso, PnR, Transition, Post-Up, Spot-Up, etc.", df)

time.sleep(1)

# 2k. LeagueDashPtStats — tracking stats (speed, distance, touches, passes, etc.)
ldpts = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Team', pt_measure_type='SpeedDistance')
df = ldpts.get_data_frames()[0]
explored['LeagueDashPtStats_SpeedDistance'] = explore("LeagueDashPtStats (Speed & Distance) — Avg speed, distance covered", df)

time.sleep(1)

ldpts2 = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Team', pt_measure_type='Passing')
df = ldpts2.get_data_frames()[0]
explored['LeagueDashPtStats_Passing'] = explore("LeagueDashPtStats (Passing) — Passes made/received, assist opportunities", df)

time.sleep(1)

ldpts3 = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Team', pt_measure_type='Possessions')
df = ldpts3.get_data_frames()[0]
explored['LeagueDashPtStats_Possessions'] = explore("LeagueDashPtStats (Possessions) — Touches, time of possession, dribbles", df)

time.sleep(1)

ldpts4 = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Team', pt_measure_type='Rebounding')
df = ldpts4.get_data_frames()[0]
explored['LeagueDashPtStats_Rebounding'] = explore("LeagueDashPtStats (Rebounding) — Reb chances, contested/uncontested", df)


📊 SynergyPlayTypes (Team, Offense) — Iso, PnR, Transition, Post-Up, Spot-Up, etc.
   Shape: (0, 22)  |  Columns: 22


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,PLAY_TYPE,TYPE_GROUPING,PERCENTILE,GP,POSS_PCT,PPP,FG_PCT,FT_POSS_PCT,TOV_POSS_PCT,SF_POSS_PCT,PLUSONE_POSS_PCT,SCORE_POSS_PCT,EFG_PCT,POSS,PTS,FGM,FGA,FGMX



📋 Data Types:


SEASON_ID            object
TEAM_ID              object
TEAM_ABBREVIATION    object
TEAM_NAME            object
PLAY_TYPE            object
TYPE_GROUPING        object
PERCENTILE           object
GP                   object
POSS_PCT             object
PPP                  object
FG_PCT               object
FT_POSS_PCT          object
TOV_POSS_PCT         object
SF_POSS_PCT          object
PLUSONE_POSS_PCT     object
SCORE_POSS_PCT       object
EFG_PCT              object
POSS                 object
PTS                  object
FGM                  object
FGA                  object
FGMX                 object
dtype: object


📊 LeagueDashPtStats (Speed & Distance) — Avg speed, distance covered
   Shape: (30, 14)  |  Columns: 14


...

TEAM_ID                  int64
TEAM_ABBREVIATION          str
TEAM_NAME                  str
GP                       int64
W                        int64
L                        int64
MIN                    float64
POINTS                   int64
TOUCHES                  int64
FRONT_CT_TOUCHES         int64
TIME_OF_POSS           float64
AVG_SEC_PER_TOUCH      float64
AVG_DRIB_PER_TOUCH     float64
PTS_PER_TOUCH          float64
ELBOW_TOUCHES            int64
POST_TOUCHES             int64
PAINT_TOUCHES            int64
PTS_PER_ELBOW_TOUCH    float64
PTS_PER_POST_TOUCH     float64
PTS_PER_PAINT_TOUCH    float64
dtype: object


📊 LeagueDashPtStats (Rebounding) — Reb chances, contested/uncontested
   Shape: (30, 34)  |  Columns: 34


,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GP,W,L,MIN,OREB,OREB_CONTEST,OREB_UNCONTEST,OREB_CONTEST_PCT,OREB_CHANCES,OREB_CHANCE_PCT,OREB_CHANCE_DEFER,OREB_CHANCE_PCT_ADJ,AVG_OREB_DIST,DREB,DREB_CONTEST,DREB_UNCONTEST,DREB_CONTEST_PCT,DREB_CHANCES,DREB_CHANCE_PCT,DREB_CHANCE_DEFER,DREB_CHANCE_PCT_ADJ,AVG_DREB_DIST,REB,REB_CONTEST,REB_UNCONTEST,REB_CONTEST_PCT,REB_CHANCES,REB_CHANCE_PCT,REB_CHANCE_DEFER,REB_CHANCE_PCT_ADJ,AVG_REB_DIST
0,1610612737,ATL,Atlanta Hawks,82,40,42,18926.0,974,565,403,0.580,2107,0.462,72,0.479,7.0,2675,628,2030,0.235,4356,0.614,388,0.674,6.3,3649,1193,2433,0.327,6309,0.578,460,0.624,6.5
1,1610612738,BOS,Boston Celtics,82,61,21,19118.0,922,488,430,0.529,2075,0.444,76,0.461,8.8,2761,712,2031,0.258,4607,0.599,439,0.662,6.5,3683,1200,2461,0.326,6555,0.562,515,0.610,7.0
2,1610612751,BKN,Brooklyn Nets,82,26,56,19171.0,894,484,405,0.541,2172,0.412,81,0.428,8.0,2493,621,1866,0.249,4171,0.598,418,0.664,6.3,3387,1105,2271,0.326,6169,0.549,499,0.597,6.7



📋 Data Types:


TEAM_ID                  int64
TEAM_ABBREVIATION          str
TEAM_NAME                  str
GP                       int64
W                        int64
L                        int64
MIN                    float64
OREB                     int64
OREB_CONTEST             int64
OREB_UNCONTEST           int64
OREB_CONTEST_PCT       float64
OREB_CHANCES             int64
OREB_CHANCE_PCT        float64
OREB_CHANCE_DEFER        int64
OREB_CHANCE_PCT_ADJ    float64
AVG_OREB_DIST          float64
DREB                     int64
DREB_CONTEST             int64
DREB_UNCONTEST           int64
DREB_CONTEST_PCT       float64
DREB_CHANCES             int64
DREB_CHANCE_PCT        float64
DREB_CHANCE_DEFER        int64
DREB_CHANCE_PCT_ADJ    float64
AVG_DREB_DIST          float64
REB                      int64
REB_CONTEST              int64
REB_UNCONTEST            int64
REB_CONTEST_PCT        float64
REB_CHANCES              int64
REB_CHANCE_PCT         float64
REB_CHANCE_DEFER         int64
REB_CHAN

## 3. Player-Level Endpoints
Endpoints that return individual player stats — the core of the **Player Model**.

We'll use **LeBron James (2544)** as a sample player.

In [11]:
from nba_api.stats.endpoints import (
    playergamelog, playergamelogs, playercareerstats, playerestimatedmetrics,
    commonplayerinfo, leaguedashplayerstats, leaguedashplayerclutch,
    leaguedashplayerbiostats, leaguehustlestatsplayer,
    leaguedashplayerptshot, leaguedashptdefend,
    playerdashboardbygeneralsplits, playerdashptpass,
    playerdashptreb, playerdashptshots, playerdashptshotdefend,
    playerprofilev2, commonteamroster
)

PLAYER_ID = 2544  # LeBron James

In [12]:
# 3a. PlayerGameLog — per-game stats for a single player
time.sleep(1)
pgl = playergamelog.PlayerGameLog(player_id=PLAYER_ID, season=SEASON)
df = pgl.get_data_frames()[0]
explored['PlayerGameLog'] = explore("PlayerGameLog — Per-game player stats", df)

time.sleep(1)

# 3b. CommonPlayerInfo — biographical data (height, weight, position, draft info)
cpi = commonplayerinfo.CommonPlayerInfo(player_id=PLAYER_ID)
df = cpi.get_data_frames()[0]
explored['CommonPlayerInfo'] = explore("CommonPlayerInfo — Bio/position/draft info", df)

time.sleep(1)

# 3c. PlayerCareerStats — season-by-season career totals
pcs = playercareerstats.PlayerCareerStats(player_id=PLAYER_ID)
dfs_career = pcs.get_data_frames()
explored['PlayerCareerStats_RegSeason'] = explore("PlayerCareerStats — Regular Season Totals", dfs_career[0])


📊 PlayerGameLog — Per-game player stats
   Shape: (70, 27)  |  Columns: 27


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,2544,0022401185,"Apr 11, 2025",LAL vs. HOU,W,22,6,11,0.545,1,4,0.25,1,1,1.000,0,4,4,8,1,0,1,1,14,5,1
1,22024,2544,0022401159,"Apr 09, 2025",LAL @ DAL,W,36,11,20,0.550,0,4,0.00,5,6,0.833,2,5,7,3,1,0,2,2,27,18,1
2,22024,2544,0022401153,"Apr 08, 2025",LAL @ OKC,L,35,8,19,0.421,3,6,0.50,9,10,0.900,0,7,7,3,1,0,6,1,28,-23,1



📋 Data Types:


SEASON_ID              str
Player_ID            int64
Game_ID                str
GAME_DATE              str
MATCHUP                str
WL                     str
MIN                  int64
FGM                  int64
FGA                  int64
FG_PCT             float64
FG3M                 int64
FG3A                 int64
FG3_PCT            float64
FTM                  int64
FTA                  int64
FT_PCT             float64
OREB                 int64
DREB                 int64
REB                  int64
AST                  int64
STL                  int64
BLK                  int64
TOV                  int64
PF                   int64
PTS                  int64
PLUS_MINUS           int64
VIDEO_AVAILABLE      int64
dtype: object


📊 CommonPlayerInfo — Bio/position/draft info
   Shape: (1, 33)  |  Columns: 33


...

PERSON_ID                           int64
FIRST_NAME                            str
LAST_NAME                             str
DISPLAY_FIRST_LAST                    str
DISPLAY_LAST_COMMA_FIRST              str
DISPLAY_FI_LAST                       str
PLAYER_SLUG                           str
BIRTHDATE                             str
SCHOOL                                str
COUNTRY                               str
LAST_AFFILIATION                      str
HEIGHT                                str
WEIGHT                                str
SEASON_EXP                          int64
JERSEY                                str
POSITION                              str
ROSTERSTATUS                          str
GAMES_PLAYED_CURRENT_SEASON_FLAG      str
TEAM_ID                             int64
TEAM_NAME                             str
TEAM_ABBREVIATION                     str
TEAM_CODE                             str
TEAM_CITY                             str
PLAYERCODE                        


📊 PlayerCareerStats — Regular Season Totals
   Shape: (23, 27)  |  Columns: 27


,PLAYER_ID,SEASON_ID,LEAGUE_ID,TEAM_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,2544,2003-04,00,1610612739,CLE,19.0,79,79,3120,622,1492,0.417,63,217,0.290,347,460,0.754,99,333,432,465,130,58,273,149,1654
1,2544,2004-05,00,1610612739,CLE,20.0,80,80,3388,795,1684,0.472,108,308,0.351,477,636,0.750,111,477,588,577,177,52,262,146,2175
2,2544,2005-06,00,1610612739,CLE,21.0,79,79,3361,875,1823,0.480,127,379,0.335,601,814,0.738,75,481,556,521,123,66,260,181,2478



📋 Data Types:


PLAYER_ID              int64
SEASON_ID                str
LEAGUE_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
PLAYER_AGE           float64
GP                     int64
GS                     int64
MIN                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PTS                    int64
dtype: object

In [13]:
# 3d. LeagueDashPlayerStats — league-wide player stats (Base + Advanced)
time.sleep(1)
ldps_base = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON, measure_type_detailed_defense='Base')
df = ldps_base.get_data_frames()[0]
explored['LeagueDashPlayerStats_Base'] = explore("LeagueDashPlayerStats (Base) — All players traditional stats", df)

time.sleep(1)

ldps_adv = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON, measure_type_detailed_defense='Advanced')
df = ldps_adv.get_data_frames()[0]
explored['LeagueDashPlayerStats_Advanced'] = explore("LeagueDashPlayerStats (Advanced) — OFF/DEF rating, USG%, TS%, PIE", df)


📊 LeagueDashPlayerStats (Base) — All players traditional stats
   Shape: (569, 67)  |  Columns: 67


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,26,14,12,0.538,486.410000,80,190,0.421,33,101,0.327,43,63,0.683,20,66,86,31,15,13,6,12,44,38,236,-17,427.7,2,0,424.0,424,364,146,239,379,352,340,388,282,281,319,281,267,419,364,359,368,397,414,397,383,241,200,327,337,299,375,159,44,364,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.0,73,44,29,0.603,1659.141667,182,424,0.429,155,363,0.427,22,27,0.815,18,156,174,108,40,37,7,3,157,50,541,227,1003.8,0,0,1066.0,86,62,362,164,152,231,220,353,58,77,37,364,372,180,381,227,261,215,286,235,370,99,498,290,214,62,247,281,44,225,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,29,8,21,0.276,638.386667,82,213,0.385,24,90,0.267,32,37,0.865,8,51,59,76,35,12,3,18,50,27,220,-152,414.8,0,0,409.0,412,445,253,473,338,348,325,472,319,294,425,312,335,87,455,386,410,281,313,406,441,312,217,369,347,495,379,281,44,372,2



📋 Data Types:


PLAYER_ID                int64
PLAYER_NAME                str
NICKNAME                   str
TEAM_ID                  int64
TEAM_ABBREVIATION          str
                         ...  
NBA_FANTASY_PTS_RANK     int64
DD2_RANK                 int64
TD3_RANK                 int64
WNBA_FANTASY_PTS_RANK    int64
TEAM_COUNT               int64
Length: 67, dtype: object


📊 LeagueDashPlayerStats (Advanced) — OFF/DEF rating, USG%, TS%, PIE
   Shape: (569, 79)  |  Columns: 79


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM,FGA,FGM_PG,FGA_PG,FG_PCT,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,E_OFF_RATING_RANK,OFF_RATING_RANK,sp_work_OFF_RATING_RANK,E_DEF_RATING_RANK,DEF_RATING_RANK,sp_work_DEF_RATING_RANK,E_NET_RATING_RANK,NET_RATING_RANK,sp_work_NET_RATING_RANK,AST_PCT_RANK,AST_TO_RANK,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,E_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,USG_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,26,14,12,0.538,18.7,109.1,109.7,109.7,112.6,111.1,111.1,-3.5,-1.4,-1.4,0.093,2.07,11.7,0.036,0.128,0.080,5.7,5.7,0.508,0.542,0.189,0.193,103.30,103.71,86.43,103.71,0.082,1050,80,190,3.1,7.3,0.421,424,364,146,239,305,320,310,310,350,277,277,353,283,283,384,186,461,315,249,291,50,51,368,365,210,217,119,122,122,329,352,340,248,228,388,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.0,73,44,29,0.603,22.7,113.5,114.0,114.0,106.9,107.6,107.6,6.6,6.4,6.4,0.086,2.70,18.6,0.011,0.087,0.051,6.9,6.9,0.612,0.621,0.123,0.127,101.39,100.84,84.03,100.84,0.058,3487,182,424,2.5,5.8,0.429,86,62,362,164,220,150,149,149,110,122,122,85,94,94,406,90,252,532,442,489,85,85,62,87,496,499,280,312,312,484,231,220,310,298,353,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,29,8,21,0.276,22.0,103.5,104.1,104.1,114.8,115.6,115.6,-11.3,-11.5,-11.5,0.178,2.17,22.3,0.011,0.073,0.042,10.3,10.3,0.441,0.480,0.173,0.178,101.46,100.83,84.02,100.83,0.053,1342,82,213,2.8,7.3,0.385,412,445,253,473,232,478,462,462,448,453,453,492,497,497,168,154,147,529,515,544,330,335,493,490,272,276,271,313,313,503,348,325,275,227,472,2



📋 Data Types:


PLAYER_ID            int64
PLAYER_NAME            str
NICKNAME               str
TEAM_ID              int64
TEAM_ABBREVIATION      str
                     ...  
FGA_RANK             int64
FGM_PG_RANK          int64
FGA_PG_RANK          int64
FG_PCT_RANK          int64
TEAM_COUNT           int64
Length: 79, dtype: object

In [14]:
# 3e. PlayerEstimatedMetrics — estimated advanced stats per player (E_OFF_RATING, etc.)
time.sleep(1)
pem = playerestimatedmetrics.PlayerEstimatedMetrics(season=SEASON)
df = pem.get_data_frames()[0]
explored['PlayerEstimatedMetrics'] = explore("PlayerEstimatedMetrics — Estimated OFF/DEF rating, pace, W%", df)

time.sleep(1)

# 3f. LeagueDashPlayerBioStats — height, weight, age distribution league-wide
ldpbs = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season=SEASON)
df = ldpbs.get_data_frames()[0]
explored['LeagueDashPlayerBioStats'] = explore("LeagueDashPlayerBioStats — Height/Weight/Age/Experience per player", df)

time.sleep(1)

# 3g. LeagueHustleStatsPlayer — individual hustle metrics
lhsp = leaguehustlestatsplayer.LeagueHustleStatsPlayer(season=SEASON)
df = lhsp.get_data_frames()[0]
explored['LeagueHustleStatsPlayer'] = explore("LeagueHustleStatsPlayer — Player hustle (deflections, loose balls, etc.)", df)

time.sleep(1)

# 3h. LeagueDashPlayerClutch — player clutch performance
ldpc = leaguedashplayerclutch.LeagueDashPlayerClutch(season=SEASON)
df = ldpc.get_data_frames()[0]
explored['LeagueDashPlayerClutch'] = explore("LeagueDashPlayerClutch — Player clutch stats", df)


📊 PlayerEstimatedMetrics — Estimated OFF/DEF rating, pace, W%
   Shape: (569, 32)  |  Columns: 32


,PLAYER_ID,PLAYER_NAME,GP,W,L,W_PCT,MIN,E_OFF_RATING,E_DEF_RATING,E_NET_RATING,E_AST_RATIO,E_OREB_PCT,E_DREB_PCT,E_REB_PCT,E_TOV_PCT,E_USG_PCT,E_PACE,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,E_OFF_RATING_RANK,E_DEF_RATING_RANK,E_NET_RATING_RANK,E_AST_RATIO_RANK,E_OREB_PCT_RANK,E_DREB_PCT_RANK,E_REB_PCT_RANK,E_TOV_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK
0,1627736,Malik Beasley,82,44,38,0.537,27.8,112.1,109.5,2.6,10.3,0.023,0.079,0.051,6.127,0.222,103.64,1,62,475,240,130,211,207,173,500,464,525,529,35,128,99
1,1642267,Bub Carrington,82,18,64,0.220,30.0,102.5,117.4,-14.9,28.2,0.014,0.131,0.072,10.927,0.159,103.44,1,312,569,514,102,492,508,533,58,523,305,396,35,359,107
2,1630577,Julian Champagnie,82,34,48,0.415,23.6,112.0,113.8,-1.8,12.5,0.036,0.137,0.087,8.302,0.170,102.69,1,157,542,378,212,214,408,303,438,340,273,308,35,301,161



📋 Data Types:


PLAYER_ID              int64
PLAYER_NAME              str
GP                     int64
W                      int64
L                      int64
W_PCT                float64
MIN                  float64
E_OFF_RATING         float64
E_DEF_RATING         float64
E_NET_RATING         float64
E_AST_RATIO          float64
E_OREB_PCT           float64
E_DREB_PCT           float64
E_REB_PCT            float64
E_TOV_PCT            float64
E_USG_PCT            float64
E_PACE               float64
GP_RANK                int64
W_RANK                 int64
L_RANK                 int64
W_PCT_RANK             int64
MIN_RANK               int64
E_OFF_RATING_RANK      int64
E_DEF_RATING_RANK      int64
E_NET_RATING_RANK      int64
E_AST_RATIO_RANK       int64
E_OREB_PCT_RANK        int64
E_DREB_PCT_RANK        int64
E_REB_PCT_RANK         int64
E_TOV_PCT_RANK         int64
E_USG_PCT_RANK         int64
E_PACE_RANK            int64
dtype: object


📊 LeagueDashPlayerBioStats — Height/Weight/Age/Experience per player
   Shape: (569, 23)  |  Columns: 23


...

PLAYER_ID                          int64
PLAYER_NAME                          str
TEAM_ID                            int64
TEAM_ABBREVIATION                    str
AGE                              float64
G                                  int64
MIN                              float64
CONTESTED_SHOTS                    int64
CONTESTED_SHOTS_2PT                int64
CONTESTED_SHOTS_3PT                int64
DEFLECTIONS                        int64
CHARGES_DRAWN                      int64
SCREEN_ASSISTS                     int64
SCREEN_AST_PTS                     int64
OFF_LOOSE_BALLS_RECOVERED          int64
DEF_LOOSE_BALLS_RECOVERED          int64
LOOSE_BALLS_RECOVERED              int64
PCT_LOOSE_BALLS_RECOVERED_OFF    float64
PCT_LOOSE_BALLS_RECOVERED_DEF    float64
OFF_BOXOUTS                        int64
DEF_BOXOUTS                        int64
BOX_OUT_PLAYER_TEAM_REBS           int64
BOX_OUT_PLAYER_REBS                int64
BOX_OUTS                           int64
PCT_BOX_OUTS_OFF


📊 LeagueDashPlayerClutch — Player clutch stats
   Shape: (498, 68)  |  Columns: 68


,GROUP_SET,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,Players,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,3,1,2,0.333,12.433333,3,5,0.600,2,4,0.500,0,0,0.000,0,2,2,2,0,0,0,0,0,0,8,-8,13.4,0,0,14.0,380,375,105,372,329,239,275,82,140,189,46,307,313,307,302,278,308,195,291,263,208,1,1,339,258,339,293,216,27,286,1
1,Players,1631260,AJ Green,AJ,1610612749,MIL,25.0,25,13,12,0.520,72.740000,6,11,0.545,6,11,0.545,2,3,0.667,0,4,4,4,0,1,0,0,8,2,20,-15,33.8,0,0,36.0,103,96,372,199,122,160,196,110,40,91,41,210,213,208,302,207,253,128,291,157,208,1,374,215,157,410,200,216,27,176,1
2,Players,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,5,3,2,0.600,12.293333,3,5,0.600,1,1,1.000,0,0,0.000,0,0,0,1,0,0,0,0,2,0,7,-17,8.5,0,0,9.0,342,304,105,122,330,239,275,82,191,276,1,307,313,307,302,382,400,250,291,263,208,1,185,339,272,421,333,216,27,318,1



📋 Data Types:


GROUP_SET                  str
PLAYER_ID                int64
PLAYER_NAME                str
NICKNAME                   str
TEAM_ID                  int64
                         ...  
NBA_FANTASY_PTS_RANK     int64
DD2_RANK                 int64
TD3_RANK                 int64
WNBA_FANTASY_PTS_RANK    int64
TEAM_COUNT               int64
Length: 68, dtype: object

In [15]:
# 3i. Player tracking stats (passing, rebounding, shooting)
time.sleep(1)
# Player Passing Tracking
ldpts_pp = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Player', pt_measure_type='Passing')
df = ldpts_pp.get_data_frames()[0]
explored['LeagueDashPtStats_Player_Passing'] = explore("LeagueDashPtStats (Player, Passing) — Passes, potential assists", df)

time.sleep(1)
# Player Possessions Tracking
ldpts_pos = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Player', pt_measure_type='Possessions')
df = ldpts_pos.get_data_frames()[0]
explored['LeagueDashPtStats_Player_Possessions'] = explore("LeagueDashPtStats (Player, Possessions) — Touches, time of poss, dribbles", df)

time.sleep(1)
# Player Speed & Distance
ldpts_sd = leaguedashptstats.LeagueDashPtStats(season=SEASON, player_or_team='Player', pt_measure_type='SpeedDistance')
df = ldpts_sd.get_data_frames()[0]
explored['LeagueDashPtStats_Player_SpeedDist'] = explore("LeagueDashPtStats (Player, SpeedDistance) — Avg speed, miles covered", df)


📊 LeagueDashPtStats (Player, Passing) — Passes, potential assists
   Shape: (569, 18)  |  Columns: 18


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,PASSES_MADE,PASSES_RECEIVED,AST,FT_AST,SECONDARY_AST,POTENTIAL_AST,AST_PTS_CREATED,AST_ADJ,AST_TO_PASS_PCT,AST_TO_PASS_PCT_ADJ
0,1630639,A.J. Lawson,1610612761,TOR,26,14,12,486.0,551,509,31,5,7,69,85,43,0.056,0.078
1,1631260,AJ Green,1610612749,MIL,73,44,29,1659.0,1622,1391,108,8,14,211,298,129,0.067,0.080
2,1642358,AJ Johnson,1610612764,WAS,29,8,21,638.0,860,940,76,5,6,152,196,87,0.088,0.101



📋 Data Types:


PLAYER_ID                int64
PLAYER_NAME                str
TEAM_ID                  int64
TEAM_ABBREVIATION          str
GP                       int64
W                        int64
L                        int64
MIN                    float64
PASSES_MADE              int64
PASSES_RECEIVED          int64
AST                      int64
FT_AST                   int64
SECONDARY_AST            int64
POTENTIAL_AST            int64
AST_PTS_CREATED          int64
AST_ADJ                  int64
AST_TO_PASS_PCT        float64
AST_TO_PASS_PCT_ADJ    float64
dtype: object


📊 LeagueDashPtStats (Player, Possessions) — Touches, time of poss, dribbles
   Shape: (569, 21)  |  Columns: 21


...

PLAYER_ID                int64
PLAYER_NAME                str
TEAM_ID                  int64
TEAM_ABBREVIATION          str
GP                       int64
W                        int64
L                        int64
MIN                    float64
POINTS                   int64
TOUCHES                  int64
FRONT_CT_TOUCHES         int64
TIME_OF_POSS           float64
AVG_SEC_PER_TOUCH      float64
AVG_DRIB_PER_TOUCH     float64
PTS_PER_TOUCH          float64
ELBOW_TOUCHES            int64
POST_TOUCHES             int64
PAINT_TOUCHES            int64
PTS_PER_ELBOW_TOUCH    float64
PTS_PER_POST_TOUCH     float64
PTS_PER_PAINT_TOUCH    float64
dtype: object


📊 LeagueDashPtStats (Player, SpeedDistance) — Avg speed, miles covered
   Shape: (569, 15)  |  Columns: 15


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,DIST_FEET,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF,AVG_SPEED,AVG_SPEED_OFF,AVG_SPEED_DEF
0,1630639,A.J. Lawson,1610612761,TOR,26,14,12,486.0,220282,41.7,21.7,20.0,4.76,5.02,4.51
1,1631260,AJ Green,1610612749,MIL,73,44,29,1659.0,696965,132.0,71.6,60.4,4.47,4.89,4.06
2,1642358,AJ Johnson,1610612764,WAS,29,8,21,638.0,270166,51.2,27.9,23.3,4.49,4.91,4.08



📋 Data Types:


PLAYER_ID              int64
PLAYER_NAME              str
TEAM_ID                int64
TEAM_ABBREVIATION        str
GP                     int64
W                      int64
L                      int64
MIN                  float64
DIST_FEET              int64
DIST_MILES           float64
DIST_MILES_OFF       float64
DIST_MILES_DEF       float64
AVG_SPEED            float64
AVG_SPEED_OFF        float64
AVG_SPEED_DEF        float64
dtype: object

## 4. Game-Level & Box Score Endpoints
Per-game data that forms the training labels and provides detailed box scores.

We'll use a sample Game ID from the 2024-25 season.

In [16]:
from nba_api.stats.endpoints import (
    leaguegamelog, leaguegamefinder,
    boxscoretraditionalv3, boxscoreadvancedv3, boxscorehustlev2,
    boxscorefourfactorsv3, boxscoremiscv3, boxscoreplayertrackv3,
    boxscorescoringv3, boxscoreusagev3, boxscoresummaryv2,
    boxscorematchupsv3,
    winprobabilitypbp, playbyplayv3,
    commonteamroster, teamplayeronoffdetails,
    leaguedashlineups, leagueseasonmatchups
)

# Get a sample game ID from the Lakers' game log
GAME_ID = explored['TeamGameLog'].iloc[0]['Game_ID']
print(f"Sample Game ID: {GAME_ID}")

Sample Game ID: 0022401199


In [17]:
# 4a. LeagueGameLog — all games in a season (team-level)
time.sleep(1)
lgl = leaguegamelog.LeagueGameLog(season=SEASON, player_or_team_abbreviation='T')
df = lgl.get_data_frames()[0]
explored['LeagueGameLog_Team'] = explore("LeagueGameLog (Team) — Every game's team-level result", df)

time.sleep(1)

# 4b. BoxScoreTraditionalV3 — traditional per-player box score for a game
bst = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=GAME_ID)
dfs_bs = bst.get_data_frames()
explored['BoxScoreTraditionalV3_Player'] = explore("BoxScoreTraditionalV3 (Player) — Per-player box score", dfs_bs[0])
explored['BoxScoreTraditionalV3_Team'] = explore("BoxScoreTraditionalV3 (Team) — Team totals for game", dfs_bs[1])

time.sleep(1)

# 4c. BoxScoreAdvancedV3 — advanced per-player box score
bsa = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=GAME_ID)
dfs_bsa = bsa.get_data_frames()
explored['BoxScoreAdvancedV3_Player'] = explore("BoxScoreAdvancedV3 (Player) — OFF/DEF RTG, USG%, TS% per player per game", dfs_bsa[0])


📊 LeagueGameLog (Team) — Every game's team-level result
   Shape: (2460, 29)  |  Columns: 29


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,95,0.505,29,61,0.475,7,8,0.875,11,29,40,33,6,3,4,15,132,23,1
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,95,0.442,5,30,0.167,21,25,0.840,15,31,46,22,7,8,7,22,110,7,1



📋 Data Types:


SEASON_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                  str
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PTS                    int64
PLUS_MINUS             int64
VIDEO_AVAILABLE        int64
dtype: object


📊 BoxScoreTraditionalV3 (Player) — Per-player box score
   Shape: (26, 34)  |  Columns: 34


...

gameId                         str
teamId                       int64
teamCity                       str
teamName                       str
teamTricode                    str
teamSlug                       str
minutes                        str
fieldGoalsMade               int64
fieldGoalsAttempted          int64
fieldGoalsPercentage       float64
threePointersMade            int64
threePointersAttempted       int64
threePointersPercentage    float64
freeThrowsMade               int64
freeThrowsAttempted          int64
freeThrowsPercentage       float64
reboundsOffensive            int64
reboundsDefensive            int64
reboundsTotal                int64
assists                      int64
steals                       int64
blocks                       int64
turnovers                    int64
foulsPersonal                int64
points                       int64
startersBench                  str
dtype: object


📊 BoxScoreAdvancedV3 (Player) — OFF/DEF RTG, USG%, TS% per player per game
   Shape: (26, 37)  |  Columns: 37


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,defensiveRating,estimatedNetRating,netRating,assistPercentage,assistToTurnover,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,1642261,Dalton,Knecht,D. Knecht,dalton-knecht,F,,,37:22,84.9,84.9,123.3,123.3,-38.4,-38.4,0.000,0.00,0.0,0.045,0.120,0.085,8.0,0.548,0.593,0.278,0.278,93.77,93.77,78.14,73.0,0.180
1,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,202693,Markieff,Morris,M. Morris,markieff-morris,F,,,32:48,91.9,91.9,121.3,121.3,-29.4,-29.4,0.353,3.00,33.3,0.000,0.100,0.056,11.1,0.550,0.550,0.162,0.162,90.00,90.00,75.00,62.0,0.105
2,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,203458,Alex,Len,A. Len,alex-len,C,,,13:33,92.3,92.3,144.4,144.4,-52.1,-52.1,0.143,0.33,12.5,0.000,0.083,0.059,37.5,0.500,0.500,0.250,0.250,93.87,93.87,78.23,26.0,-0.080



📋 Data Types:


gameId                              str
teamId                            int64
teamCity                            str
teamName                            str
teamTricode                         str
teamSlug                            str
personId                          int64
firstName                           str
familyName                          str
nameI                               str
playerSlug                          str
position                            str
comment                             str
jerseyNum                           str
minutes                             str
estimatedOffensiveRating        float64
offensiveRating                 float64
estimatedDefensiveRating        float64
defensiveRating                 float64
estimatedNetRating              float64
netRating                       float64
assistPercentage                float64
assistToTurnover                float64
assistRatio                     float64
offensiveReboundPercentage      float64


In [19]:
# 4d. BoxScoreHustleV2 — per-player hustle stats for a single game
time.sleep(1)
bsh = boxscorehustlev2.BoxScoreHustleV2(game_id=GAME_ID)
df = bsh.get_data_frames()[0]
explored['BoxScoreHustleV2'] = explore("BoxScoreHustleV2 — Hustle stats per player per game", df)

time.sleep(1)

# 4e. BoxScoreMatchupsV3 — defensive matchups for a game
bsm = boxscorematchupsv3.BoxScoreMatchupsV3(game_id=GAME_ID)
df = bsm.get_data_frames()[0]
explored['BoxScoreMatchupsV3'] = explore("BoxScoreMatchupsV3 — Who guarded whom, matchup stats", df)

time.sleep(1)

# 4f. BoxScoreSummaryV2 — game summary info (line score, officials, etc.)
bss = boxscoresummaryv2.BoxScoreSummaryV2(game_id=GAME_ID)
dfs_bss = bss.get_data_frames()
explored['BoxScoreSummary_GameSummary'] = explore("BoxScoreSummaryV2 — Game Summary (date, arena, attendance)", dfs_bss[0])
explored['BoxScoreSummary_LineScore'] = explore("BoxScoreSummaryV2 — Line Score (quarter-by-quarter)", dfs_bss[5])

time.sleep(1)

# 4g. WinProbabilityPBP — play-by-play with win probability
# Use an older completed game since newest games may not have data
try:
    wpp = winprobabilitypbp.WinProbabilityPBP(game_id='0022400062')
    df = wpp.get_data_frames()[0]
    explored['WinProbabilityPBP'] = explore("WinProbabilityPBP — Play-by-play win probability", df)
except Exception as e:
    print(f"⚠️ WinProbabilityPBP failed: {e}")
    print("Columns expected: GAME_ID, EVENT_NUM, HOME_PCT, VISITOR_PCT, HOME_PTS, VISITOR_PTS, etc.")


📊 BoxScoreHustleV2 — Hustle stats per player per game
   Shape: (17, 31)  |  Columns: 31


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,points,contestedShots,contestedShots2pt,contestedShots3pt,deflections,chargesDrawn,screenAssists,screenAssistPoints,looseBallsRecoveredOffensive,looseBallsRecoveredDefensive,looseBallsRecoveredTotal,offensiveBoxOuts,defensiveBoxOuts,boxOutPlayerTeamRebounds,boxOutPlayerRebounds,boxOuts
0,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,1642261,Dalton,Knecht,D. Knecht,dalton-knecht,F,,4,37:22,27,3,2,1,0,0,0,0,0,1,1,0,1,1,1,1
1,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,202693,Markieff,Morris,M. Morris,markieff-morris,F,,88,32:48,11,4,1,3,2,0,2,5,0,0,0,0,1,1,1,1
2,0022401199,1610612747,Los Angeles,Lakers,LAL,lakers,203458,Alex,Len,A. Len,alex-len,C,,27,13:33,4,5,5,0,2,0,1,2,1,1,2,0,0,0,0,0



📋 Data Types:


gameId                            str
teamId                          int64
teamCity                          str
teamName                          str
teamTricode                       str
teamSlug                          str
personId                        int64
firstName                         str
familyName                        str
nameI                             str
playerSlug                        str
position                          str
comment                           str
jerseyNum                         str
minutes                           str
points                          int64
contestedShots                  int64
contestedShots2pt               int64
contestedShots3pt               int64
deflections                     int64
chargesDrawn                    int64
screenAssists                   int64
screenAssistPoints              int64
looseBallsRecoveredOffensive    int64
looseBallsRecoveredDefensive    int64
looseBallsRecoveredTotal        int64
offensiveBox


📊 BoxScoreMatchupsV3 — Who guarded whom, matchup stats
   Shape: (139, 46)  |  Columns: 46


...


📊 BoxScoreSummaryV2 — Line Score (quarter-by-quarter)
   Shape: (2, 23)  |  Columns: 23


,GAME_DATE_EST,GAME_SEQUENCE,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY_NAME,TEAM_NICKNAME,TEAM_WINS_LOSSES,PTS_QTR1,PTS_QTR2,PTS_QTR3,PTS_QTR4,PTS_OT1,PTS_OT2,PTS_OT3,PTS_OT4,PTS_OT5,PTS_OT6,PTS_OT7,PTS_OT8,PTS_OT9,PTS_OT10,PTS
0,2025-04-13T00:00:00,14,0022401199,1610612747,LAL,Los Angeles,Lakers,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,2025-04-13T00:00:00,14,0022401199,1610612757,POR,Portland,Trail Blazers,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None



📋 Data Types:


GAME_DATE_EST           str
GAME_SEQUENCE         int64
GAME_ID                 str
TEAM_ID               int64
TEAM_ABBREVIATION       str
TEAM_CITY_NAME          str
TEAM_NICKNAME           str
TEAM_WINS_LOSSES     object
PTS_QTR1             object
PTS_QTR2             object
PTS_QTR3             object
PTS_QTR4             object
PTS_OT1              object
PTS_OT2              object
PTS_OT3              object
PTS_OT4              object
PTS_OT5              object
PTS_OT6              object
PTS_OT7              object
PTS_OT8              object
PTS_OT9              object
PTS_OT10             object
PTS                  object
dtype: object

⚠️ WinProbabilityPBP failed: Expecting value: line 1 column 1 (char 0)
Columns expected: GAME_ID, EVENT_NUM, HOME_PCT, VISITOR_PCT, HOME_PTS, VISITOR_PTS, etc.


## 5. Lineup, On/Off Court & Matchup Endpoints
These capture lineup chemistry, player on/off impact, and season-level matchup data.

In [20]:
from nba_api.stats.endpoints import (
    commonteamroster, leaguedashlineups, leagueseasonmatchups,
    leaguelineupviz, teamdashlineups, teamplayeronoffsummary,
    shotchartdetail, playbyplayv3
)

# 5a. CommonTeamRoster — who's on the team, position, height, weight, age
time.sleep(1)
ctr = commonteamroster.CommonTeamRoster(team_id=TEAM_ID, season=SEASON)
dfs_ctr = ctr.get_data_frames()
explored['CommonTeamRoster'] = explore("CommonTeamRoster — Players, position, height, weight, age", dfs_ctr[0])
if len(dfs_ctr) > 1:
    explored['CommonTeamRoster_Coaches'] = explore("CommonTeamRoster — Coaches", dfs_ctr[1])

time.sleep(1)

# 5b. TeamPlayerOnOffDetails — team performance with/without each player
tpoo = teamplayeronoffdetails.TeamPlayerOnOffDetails(team_id=TEAM_ID, season=SEASON)
dfs_oo = tpoo.get_data_frames()
explored['TeamPlayerOnOff_On'] = explore("TeamPlayerOnOffDetails — On Court stats per player", dfs_oo[0])
explored['TeamPlayerOnOff_Off'] = explore("TeamPlayerOnOffDetails — Off Court stats per player", dfs_oo[1])

time.sleep(1)

# 5c. TeamPlayerOnOffSummary — simpler on/off splits
tpos = teamplayeronoffsummary.TeamPlayerOnOffSummary(team_id=TEAM_ID, season=SEASON)
dfs_os = tpos.get_data_frames()
explored['TeamPlayerOnOffSummary_On'] = explore("TeamPlayerOnOffSummary — On Court summary", dfs_os[0])
explored['TeamPlayerOnOffSummary_Off'] = explore("TeamPlayerOnOffSummary — Off Court summary", dfs_os[1])

time.sleep(1)

# 5d. LeagueDashLineups — 5-man lineup stats
ldl = leaguedashlineups.LeagueDashLineups(season=SEASON, group_quantity=5)
df = ldl.get_data_frames()[0]
explored['LeagueDashLineups_5Man'] = explore("LeagueDashLineups (5-Man) — Five-man lineup NET_RATING, MIN", df)

time.sleep(1)

# 5e. TeamDashLineups — team-specific lineup combos
tdl = teamdashlineups.TeamDashLineups(team_id=TEAM_ID, season=SEASON, group_quantity=5)
df = tdl.get_data_frames()[0]
explored['TeamDashLineups_5Man'] = explore("TeamDashLineups (5-Man) — Team-specific five-man lineups", df)

time.sleep(1)

# 5f. ShotChartDetail — shot-level coordinates for a player
scd = shotchartdetail.ShotChartDetail(
    team_id=TEAM_ID, player_id=PLAYER_ID, season_nullable=SEASON,
    context_measure_simple='FGA'
)
dfs_sc = scd.get_data_frames()
explored['ShotChartDetail'] = explore("ShotChartDetail — Shot x/y coordinates, zone, distance, result", dfs_sc[0])
if len(dfs_sc) > 1:
    explored['ShotChartDetail_LeagueAvg'] = explore("ShotChartDetail — League Average", dfs_sc[1])

time.sleep(1)

# 5g. PlayByPlayV3 — play-by-play event log
try:
    pbp = playbyplayv3.PlayByPlayV3(game_id=GAME_ID)
    df = pbp.get_data_frames()[0]
    explored['PlayByPlayV3'] = explore("PlayByPlayV3 — Event-level play-by-play data", df)
except Exception as e:
    print(f"⚠️  PlayByPlayV3 error: {e}")
    explored['PlayByPlayV3'] = {'note': 'endpoint failed', 'expected_cols': [
        'gameId','actionNumber','clock','period','teamId','teamTricode',
        'personId','playerNameI','xLegacy','yLegacy','shotDistance',
        'shotResult','isFieldGoal','scoreHome','scoreAway','pointsTotal',
        'actionType','subType','descriptor','qualifiers'
    ]}

time.sleep(1)

# 5h. LeagueSeasonMatchups — player vs player matchup data
try:
    lsm = leagueseasonmatchups.LeagueSeasonMatchups(season=SEASON)
    df = lsm.get_data_frames()[0]
    explored['LeagueSeasonMatchups'] = explore("LeagueSeasonMatchups — Player-vs-player season matchup stats", df)
except Exception as e:
    print(f"⚠️  LeagueSeasonMatchups error: {e}")
    explored['LeagueSeasonMatchups'] = {'note': 'endpoint failed', 'expected_cols': [
        'SEASON_ID','OFF_PLAYER_ID','OFF_PLAYER_NAME','DEF_PLAYER_ID',
        'DEF_PLAYER_NAME','GP','MATCHUP_MIN','PARTIAL_POSS',
        'PLAYER_PTS','TEAM_PTS','MATCHUP_AST','MATCHUP_TOV',
        'MATCHUP_FGM','MATCHUP_FGA','MATCHUP_FG_PCT',
        'MATCHUP_FG3M','MATCHUP_FG3A','MATCHUP_FG3_PCT'
    ]}

print("\n✅ Section 5 complete!")


📊 CommonTeamRoster — Players, position, height, weight, age
   Shape: (17, 16)  |  Columns: 16


,TeamID,SEASON,LeagueID,PLAYER,NICKNAME,PLAYER_SLUG,NUM,POSITION,HEIGHT,WEIGHT,BIRTH_DATE,AGE,EXP,SCHOOL,PLAYER_ID,HOW_ACQUIRED
0,1610612747,2024,00,Jarred Vanderbilt,Jarred,jarred-vanderbilt,2,F,6-8,214,"APR 03, 1999",26.0,6,Kentucky,1629020,Traded from UTA on 02/09/23
1,1610612747,2024,00,Dalton Knecht,Dalton,dalton-knecht,4,F,6-6,215,"APR 19, 2001",24.0,R,Tennessee,1642261,#17 Pick in 2024 Draft
2,1610612747,2024,00,Gabe Vincent,Gabe,gabe-vincent,7,G,6-2,200,"JUN 14, 1996",29.0,5,California-Santa Barbara,1629216,Signed on 07/06/23



📋 Data Types:


TeamID            int64
SEASON              str
LeagueID            str
PLAYER              str
NICKNAME            str
PLAYER_SLUG         str
NUM                 str
POSITION            str
HEIGHT              str
WEIGHT              str
BIRTH_DATE          str
AGE             float64
EXP                 str
SCHOOL              str
PLAYER_ID         int64
HOW_ACQUIRED        str
dtype: object


📊 CommonTeamRoster — Coaches
   Shape: (8, 10)  |  Columns: 10


...


📊 LeagueSeasonMatchups — Player-vs-player season matchup stats
   Shape: (141255, 27)  |  Columns: 27


,SEASON_ID,OFF_PLAYER_ID,OFF_PLAYER_NAME,DEF_PLAYER_ID,DEF_PLAYER_NAME,GP,MATCHUP_MIN,PARTIAL_POSS,PLAYER_PTS,TEAM_PTS,MATCHUP_AST,MATCHUP_TOV,MATCHUP_BLK,MATCHUP_FGM,MATCHUP_FGA,MATCHUP_FG_PCT,MATCHUP_FG3M,MATCHUP_FG3A,MATCHUP_FG3_PCT,HELP_BLK,HELP_FGM,HELP_FGA,HELP_FG_PERC,MATCHUP_FTM,MATCHUP_FTA,SFL,MATCHUP_TIME_SEC
0,22024,203076,Anthony Davis,1627734,Domantas Sabonis,4,40:58,195.2,53,210,15,7,0,17,38,0.447,1,2,0.500,0,0,0,0,18,20,10,2458.4
1,22024,203497,Rudy Gobert,1627734,Domantas Sabonis,4,39:51,186.6,21,211,4,2,1,9,11,0.818,0,0,0.000,0,0,0,0,3,4,2,2390.8
2,22024,1628973,Jalen Brunson,1630700,Dyson Daniels,4,38:03,184.6,52,196,9,5,1,19,38,0.500,4,13,0.308,0,0,0,0,10,16,6,2282.7



📋 Data Types:


SEASON_ID               str
OFF_PLAYER_ID         int64
OFF_PLAYER_NAME         str
DEF_PLAYER_ID         int64
DEF_PLAYER_NAME         str
GP                    int64
MATCHUP_MIN             str
PARTIAL_POSS        float64
PLAYER_PTS            int64
TEAM_PTS              int64
MATCHUP_AST           int64
MATCHUP_TOV           int64
MATCHUP_BLK           int64
MATCHUP_FGM           int64
MATCHUP_FGA           int64
MATCHUP_FG_PCT      float64
MATCHUP_FG3M          int64
MATCHUP_FG3A          int64
MATCHUP_FG3_PCT     float64
HELP_BLK              int64
HELP_FGM              int64
HELP_FGA              int64
HELP_FG_PERC          int64
MATCHUP_FTM           int64
MATCHUP_FTA           int64
SFL                   int64
MATCHUP_TIME_SEC    float64
dtype: object


✅ Section 5 complete!


In [22]:
# Summary of all explored endpoints
print(f"{'Endpoint':<40} {'Shape':<20} {'Columns':<8}")
print("=" * 70)
for name, info in explored.items():
    if isinstance(info, dict) and 'note' in info:
        print(f"{name:<40} {'FAILED':<20} {len(info.get('expected_cols', [])):<8}")
    elif isinstance(info, pd.DataFrame):
        print(f"{name:<40} {str(info.shape):<20} {info.shape[1]:<8}")
    else:
        print(f"{name:<40} {'unknown':<20}")
print(f"\nTotal endpoints explored: {len(explored)}")

Endpoint                                 Shape                Columns 
TeamGameLog                              (82, 27)             27      
TeamEstimatedMetrics                     (30, 30)             30      
LeagueDashTeamStats_Base                 (30, 54)             54      
LeagueDashTeamStats_Advanced             (30, 46)             46      
LeagueDashTeamStats_FourFactors          (30, 28)             28      
LeagueDashTeamStats_Opponent             (30, 54)             54      
LeagueDashTeamClutch                     (30, 54)             54      
LeagueHustleStatsTeam                    (30, 20)             20      
LeagueStandingsV3                        (30, 92)             92      
LeagueDashTeamPtShot                     (30, 18)             18      
LeagueDashPtTeamDefend                   (30, 11)             11      
TeamDashboardByGeneralSplits_Overall     (1, 55)              55      
TeamDashboardByGeneralSplits_ByLocation  (2, 55)              55      
Synerg

## 6. Feature Summary & Model Architecture Recommendations

---

### 🏀 Model 1 — Team vs Team "Playstyle" Model
**Goal:** Predict win probability when Team A faces Team B, capturing *how* each team plays.

#### Feature Groups

| Category | Source Endpoint(s) | Key Features |
|---|---|---|
| **Offensive Efficiency** | `LeagueDashTeamStats_Advanced` | `OFF_RATING`, `NET_RATING`, `TS_PCT`, `EFG_PCT`, `AST_PCT`, `AST_TO`, `AST_RATIO` |
| **Defensive Efficiency** | `LeagueDashTeamStats_Advanced`, `LeagueDashTeamStats_Opponent` | `DEF_RATING`, `OPP_FG_PCT`, `OPP_FG3_PCT`, `OPP_PTS_PAINT`, `OPP_PTS_FB`, `OPP_TOV` |
| **Four Factors** | `LeagueDashTeamStats_FourFactors` | `EFG_PCT`, `FTA_RATE`, `TM_TOV_PCT`, `OREB_PCT` (+ opponent mirror) |
| **Pace & Tempo** | `LeagueDashPtStats_SpeedDistance`, `LeagueDashPtStats_Possessions` | `AVG_SPEED`, `AVG_SPEED_OFF`, `AVG_SPEED_DEF`, `PACE`, `POSS`, `TIME_OF_POSS`, `TOUCHES` |
| **Shot Profile** | `LeagueDashTeamPtShot`, `ShotChartDetail` | `FGA_FREQUENCY` by zone, `FG_PCT` by zone, `FG2A_FREQUENCY`, `FG3A_FREQUENCY`, shot zone distributions |
| **Passing & Ball Movement** | `LeagueDashPtStats_Passing` | `PASSES_MADE`, `PASSES_RECEIVED`, `AST`, `SECONDARY_AST`, `POTENTIAL_AST`, `AST_ADJ` |
| **Rebounding** | `LeagueDashPtStats_Rebounding` | `OREB`, `DREB`, `OREB_CONTEST`, `OREB_UNCONTEST`, `DREB_CHANCES`, `REB_CHANCE_PCT` |
| **Defense Tracking** | `LeagueDashPtTeamDefend` | `D_FGA`, `D_FGM`, `D_FG_PCT`, `NORMAL_FG_PCT`, `PCT_PLUSMINUS` (vs league avg by distance) |
| **Hustle / Effort** | `LeagueHustleStatsTeam` | `CONTESTED_SHOTS`, `DEFLECTIONS`, `LOOSE_BALLS_RECOVERED`, `CHARGES_DRAWN`, `SCREEN_ASSISTS` |
| **Clutch Performance** | `LeagueDashTeamClutch` | All base stats filtered to clutch situations (last 5 min, ≤5 pts) — `W_PCT`, `FG_PCT`, `NET_RATING` |
| **Play Type Tendencies** | `SynergyPlayTypes` | `POSS_PCT` and `PPP` for Transition, Isolation, P&R Ball Handler, P&R Roll Man, Post-Up, Spot-Up, Handoff, Cut, Off-Screen |
| **Standings / Form** | `LeagueStandingsV3` | `W_PCT`, `L10` record, `STREAK`, `ConferenceGamesWon`, `HOME_RECORD`, `ROAD_RECORD`, `OT_RECORD` |
| **Home / Away Splits** | `TeamDashboardByGeneralSplits` | All stats split by `Home` vs `Away` — captures home-court advantage in playstyle |
| **Lineup Chemistry** | `LeagueDashLineups_5Man`, `TeamDashLineups_5Man` | `NET_RATING`, `PLUS_MINUS`, `MIN` for top lineups — weighted by minutes |

#### Derived / Engineered Features
- **Δ-features:** Difference between Team A and Team B for every metric (e.g., `OFF_RATING_A - DEF_RATING_B`)
- **Pace-adjusted stats:** Per-100-possessions normalization
- **Rolling form:** Last 5/10/15 game moving averages from `TeamGameLog`
- **Rest days:** Days since last game (from `TeamGameLog` dates)
- **Home/Away indicator**
- **Head-to-head history:** Prior matchup results from `LeagueGameLog`

---

### 👤 Model 2 — Player Model (feeds into Team Model)
**Goal:** Quantify each player's individual contribution across multiple dimensions, then aggregate into team-level features.

#### Feature Groups

| Category | Source Endpoint(s) | Key Features |
|---|---|---|
| **Bio / Physical** | `LeagueDashPlayerBioStats`, `CommonPlayerInfo` | `PLAYER_HEIGHT_INCHES`, `PLAYER_WEIGHT`, `AGE`, `DRAFT_ROUND`, `DRAFT_NUMBER`, `POSITION`, `COUNTRY` |
| **Traditional Stats** | `LeagueDashPlayerStats_Base`, `PlayerGameLog` | `PTS`, `REB`, `AST`, `STL`, `BLK`, `TOV`, `FG_PCT`, `FG3_PCT`, `FT_PCT`, `MIN`, `GP` |
| **Advanced Metrics** | `LeagueDashPlayerStats_Advanced` | `OFF_RATING`, `DEF_RATING`, `NET_RATING`, `PIE`, `TS_PCT`, `EFG_PCT`, `USG_PCT`, `AST_PCT`, `AST_TO`, `PACE` |
| **Estimated Metrics** | `PlayerEstimatedMetrics` | `E_OFF_RATING`, `E_DEF_RATING`, `E_NET_RATING`, `E_AST_RATIO`, `E_OREB_PCT`, `E_DREB_PCT`, `E_TOV_PCT`, `E_USG_PCT`, `E_PACE` |
| **Hustle Stats** | `LeagueHustleStatsPlayer` | `CONTESTED_SHOTS`, `DEFLECTIONS`, `LOOSE_BALLS_RECOVERED`, `CHARGES_DRAWN`, `SCREEN_ASSISTS`, `BOX_OUTS` |
| **Clutch** | `LeagueDashPlayerClutch` | All base + advanced stats in clutch situations |
| **Passing** | `LeagueDashPtStats_Player_Passing` | `PASSES_MADE`, `PASSES_RECEIVED`, `AST`, `SECONDARY_AST`, `POTENTIAL_AST`, `AST_ADJ`, `AST_TO_PASS_PCT` |
| **Ball Handling** | `LeagueDashPtStats_Player_Possessions` | `TOUCHES`, `TIME_OF_POSS`, `AVG_SEC_PER_TOUCH`, `AVG_DRIB_PER_TOUCH`, `PTS_PER_TOUCH`, `ELBOW_TOUCHES`, `PAINT_TOUCHES`, `POST_TOUCHES` |
| **Movement** | `LeagueDashPtStats_Player_SpeedDist` | `AVG_SPEED`, `AVG_SPEED_OFF`, `AVG_SPEED_DEF`, `DIST_MILES`, `DIST_MILES_OFF`, `DIST_MILES_DEF` |
| **Shot Chart** | `ShotChartDetail` | Per-zone FGA, FGM, FG% distributions — build a shot profile vector per player |
| **Career Trajectory** | `PlayerCareerStats` | Year-over-year trends in `PTS`, `REB`, `AST`, `PER`, `TS_PCT` — detect improvement/decline |
| **On/Off Impact** | `TeamPlayerOnOffSummary` | `ON_NET_RATING`, `OFF_NET_RATING`, delta = player's marginal impact on team |

#### Aggregation Strategy (Player → Team)
- **Weighted average** by projected minutes for each player on the roster
- **Top-N players:** Aggregate only the 8-player rotation
- **Positional encoding:** Group by position (G/F/C) and aggregate within position
- **Roster composition vector:** Embed the full starting-5 + bench into a fixed-size vector

---

### 🧠 Model 3 (Suggested) — Lineup Synergy / Chemistry Model
**Goal:** Capture non-linear interactions between specific player combinations.

| Source | Key Features |
|---|---|
| `LeagueDashLineups_5Man` | `NET_RATING`, `PLUS_MINUS`, `MIN`, `W_PCT` for each 5-man unit |
| `TeamDashLineups_5Man` | Team-specific lineup combos |
| `TeamPlayerOnOffDetails` | Per-player on/off court impact — enables pairwise interaction features |
| `BoxScoreMatchupsV3` | In-game player-vs-player matchup outcomes |

**Approach:** Use the player embeddings from Model 2 as inputs. For a given 5-player lineup, concatenate or attention-pool the 5 player vectors and predict lineup `NET_RATING`. This captures synergy (e.g., spacing around a post player, dual ball-handler lineups).

---

### 🎯 Model 4 (Suggested) — Matchup Advantage Model
**Goal:** Predict positional/individual matchup advantages when two specific rosters clash.

| Source | Key Features |
|---|---|
| `LeagueSeasonMatchups` | `MATCHUP_FG_PCT`, `MATCHUP_FG3_PCT`, `PLAYER_PTS`, `PARTIAL_POSS`, `MATCHUP_TOV`, `MATCHUP_BLK`, `SFL` — for every OFF_PLAYER vs DEF_PLAYER |
| `BoxScoreMatchupsV3` | Game-level matchup data |
| `LeagueDashPtTeamDefend` | Team defense by shot distance |

**Approach:** For each position matchup (e.g., Team A's PG vs Team B's PG), use historical matchup data + player embeddings to predict a per-position advantage score. Sum across all 5 positions for an overall matchup advantage.

---

### 🔄 Model 5 (Suggested) — Game State / Momentum Model
**Goal:** Capture in-game dynamics and momentum shifts.

| Source | Key Features |
|---|---|
| `PlayByPlayV3` | Event sequences — runs, lead changes, timeout patterns |
| `BoxScoreSummaryV2` | Quarter-by-quarter scores (`LineScore`) |
| `LeagueDashTeamClutch` | Clutch-specific performance |
| `WinProbabilityPBP` | Real-time win probability curves (when available) |

**Approach:** Sequential model (LSTM/Transformer) on play-by-play event sequences. Useful for live win-probability prediction and capturing momentum.

---

### 🏗️ Recommended Architecture Pipeline

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────────┐
│  Player Model   │────▶│  Team Model      │────▶│  Game Prediction    │
│  (per-player    │     │  (aggregate      │     │  (Team A vs Team B) │
│   embeddings)   │     │   player vecs +  │     │                     │
│                 │     │   team-level     │     │  Output: P(win_A)   │
└─────────────────┘     │   features)      │     └─────────────────────┘
                        └──────────────────┘               ▲
                                 ▲                         │
                        ┌────────┴────────┐      ┌────────┴────────┐
                        │ Lineup Synergy  │      │ Matchup Adv.    │
                        │ Model           │      │ Model           │
                        └─────────────────┘      └─────────────────┘
```

1. **Train Player Model** → player embedding vectors (128-d or similar)
2. **Lineup Synergy Model** → lineup-level NET_RATING prediction
3. **Aggregate** player vectors + team stats → team feature vector
4. **Matchup Advantage Model** → positional edge scores
5. **Final Game Model** → takes (team_A_vec, team_B_vec, matchup_scores, home/away, rest_days, form) → **P(Team A wins)**

---

### 📦 Key Data Volumes (2024-25 season)

| Granularity | Rows per Season | Updates |
|---|---|---|
| Team-level stats | 30 teams | Season-end |
| Player-level stats | ~570 players | Season-end |
| Game logs | ~2,460 team-games (1,230 games) | Daily |
| Box scores | ~26 player rows per game × 1,230 | Daily |
| Play-by-play | ~450 events per game × 1,230 | Daily |
| Shot charts | ~85 shots/player × 570 | Daily |
| Matchups | ~141K player-vs-player pairs | Season-end |
| 5-man lineups | ~2,000 league-wide | Season-end |